In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd
import glob
from matplotlib import pyplot as plt

In [ ]:
path = '/kaggle/input/asl-dataset/asl_dataset/asl_dataset' 

In [ ]:
fig, ax = plt.subplots(4, 9, figsize=(10, 5))

img_classes = [os.path.join(path, im) for im in os.listdir(path)]

ax = ax.flatten()

for i, img_class in enumerate(img_classes):
    img = os.listdir(img_class)[np.random.randint(0, 10)]
    img_path = os.path.join(img_class, img)
    image = cv2.imread(img_path)
    ax[i].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax[i].set_title(os.path.basename(img_class))
    ax[i].axis('off')

plt.show()

In [ ]:
for folder in os.listdir(path):
    folder_path = os.path.join(path, folder)
    image_files = glob.glob(os.path.join(folder_path, '*.jpeg'))
    num_images = len(image_files)
    print(folder, 'Number of images in folder:', num_images)

In [ ]:

data = tf.keras.preprocessing.image_dataset_from_directory(
    path,
    image_size=(64, 64),
    batch_size=32,
    validation_split=0.2,
    seed=123,
    label_mode = 'categorical',
    subset="training")

In [ ]:
train_size = int(0.8 * len(data))
val_size = int(0.2 * len(data)) + 1
train_data = data.take(train_size)
val_data = data.skip(train_size).take(val_size)
test_data = data = tf.keras.preprocessing.image_dataset_from_directory(
    path,
    image_size=(64, 64),
    batch_size=32,
    validation_split=0.2,
    seed=123,
    label_mode = 'categorical',
    subset="validation")

In [ ]:
train_data = train_data.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
cnn_model = tf.keras.Sequential([
    tf.keras.layers.experimental.preprocessing.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    
    tf.keras.layers.Flatten(),
    
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(36, activation = 'softmax')
])

In [ ]:
cnn_model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

early_stopping = tf.keras.callbacks.EarlyStopping( monitor='val_loss', 
                                                   patience=3, 
                                                   mode='min', 
                                                   restore_best_weights=True)

In [ ]:

history = cnn_model.fit(train_data, epochs=10, 
                    validation_data=val_data, 
                    callbacks=[early_stopping])

In [ ]:
epochs= []
for i in range(10):
    epochs.append(i)
    
plt.figure(figsize = (15, 10))    
plt.plot(epochs,history.history['accuracy'], label="Train")
plt.plot(epochs,history.history['val_accuracy'], label="Test")
plt.title("Train-Test Accuracy")
plt.xlabel("Number of Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize = (15, 10))   
plt.plot(epochs,history.history['loss'], label="Train")
plt.plot(epochs,history.history['val_loss'], label="Test")
plt.title("Train-Test Loss")
plt.xlabel("Number of Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
cnn_model.evaluate(test_data)